<div style="
    width: 75%;
    display: flex;
    margin: 0 auto;
    padding: 18px 20px;
    border-radius: 10px;
    align-items: center;
    margin-bottom: -20px
    box-sizing: border-box;
    justify-content: center;
">
    <img  src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg"  alt="Logo Petrobras" style="height: 55px; margin-right: 16px;">
    <span style="
        font-size: 32px;
        text-align: center;
        font-family: Arial, sans-serif;
    ">
        <b>Tutorial:</b> Modelagem MultiRotor
    </span>
</div>

---

Na aula de hoje, aprenderemos a modelar um **sistema MultiRotor**, composto por dois rotores conectados por meio de uma caixa de engrenagens. Com o objetivo de aplicar os conhecimentos adquiridos, realizaremos a montagem do sistema ilustrado abaixo. 

* **Rotor Baixa Velocidade:** Localizado na parte superior da imagem, este conjunto é formado pelo motor, um acoplamento e o seu respectivo engrenamento, e apresenta a menor velocidade de rotação do conjunto. 
* **Rotor Alta Velocidade:** Localizado na parte inferior, este conjunto é composto pelo compressor, um acoplamento e o seu respectivo engrenamento, e apresenta a maior velocidade de rotação do conjunto. 

Os dados empregados na modelagem deste sistema foram extraídos de documentos técnicos fornecidos pela **Petrobras**.

<p align="center"><img src="figs/multirrotor_completo.png" width="80%"></p>

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 32px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1. Modelagem dos elementos do rotor
    </span>
</div>

Nesta etapa, os componentes do rotor (motor, compressor, acoplamentos e engrenamentos) serão modelados individualmente. Posteriormente, utilizaremos a função `concatenate_rotor` para consolidá-los em dois subconjuntos principais. Para concluir a montagem, empregaremos a classe `MultiRotor`, que integra os rotores de baixa velocidade e alta velocidade em um único modelo dinâmico.

Daremos início ao processo importando os pacotes necessários. 

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.1. Inicialização do ambiente
    </span>
</div>

In [ ]:
import ross as rs
import numpy as np
from ross.units import Q_
from copy import deepcopy
from ross.utils import convert_6dof_to_torsional

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.2. Modelagem do motor
    </span>
</div>

Iniciaremos a modelagem pelo motor. A construção de um rotor no **ROSS** exige a definição de objetos que representem os elementos de eixo, mancais, selos e discos.

Para este componente, utilizaremos apenas elementos de eixo e mancais. No entanto, para garantir a fidelidade dinâmica do modelo, definiremos dois materiais distintos:

* **Aço (AISI 4140):** Para a estrutura do eixo propriamente dita.
* **Material das Bobinas:** Será utilizado para representar a massa e a inércia dos enrolamentos do motor, compostos majoritariamente de cobre.

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.2.1. Modelagem do eixo do motor
    </span>
</div>

Nesta etapa, definiremos a geometria base do motor. Para isso, especificamos as propriedades do material, bem como o comprimento e os diâmetros (interno e externo) de cada um dos elementos que compõem o eixo.

In [3]:
motor_shaft_material = rs.Material(name="AISI_4140", rho=7896.68, E=2e11, Poisson=0.3)

motor_shaft_length = np.array([  9.0, 150.0, 217.8, 146.0, 127.0,  22.5,  22.5,  22.5,  22.5, 239.0,  35.0,  36.2,
                                36.2,  36.2,  35.6, 75.27, 75.27, 75.27, 61.67, 61.67, 61.67,  35.0, 144.0,  22.5,
                                22.5,  22.5,  22.5, 116.0, 130.4, 304.6,  20.0, 200.0]) * 1e-3  

motor_shaft_od = np.array([ 65, 110, 125, 125, 160, 195, 195, 195, 195, 214, 215, 215, 215, 215, 215, 215, 215, 215,
                           215, 215, 215, 215, 214, 195, 195, 195, 195, 160, 125, 125, 150, 210]) * 1e-3 

motor_shaft_id = np.zeros_like(motor_shaft_length)

motor_shaft_elements = [
    rs.ShaftElement(
        L=motor_shaft_length[n],
        idl=motor_shaft_id[n],
        odl=motor_shaft_od[n],
        material=motor_shaft_material,
        n=n,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for n in range(len(motor_shaft_length))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.2.2. Modelagem das bobinas do motor
    </span>
</div>

Após definirmos a estrutura base do eixo, passaremos à modelagem dos componentes que não possuem função estrutural, mas que agregam **massa** e **inércia** ao sistema. No caso de um motor elétrico, esses elementos representam majoritariamente os enrolamentos das bobinas fixados ao rotor.

In [4]:
motor_coil_node        = np.array([   0,    5,    6,    7,    8,   10,   11,   12,   13,   14,   15,   16,   17,   18,   19,   20,   21,   23,   24,   25,   26]) 
motor_coil_mass        = np.array([  41,   60,   60,   60,   60,  955,  955,  955,  955,  955,  955,  955,  955,  955,  955,  955,  955,   60,   60,   60,   60]).astype(float)  
motor_coil_inertia     = np.array([   3, 1.64, 1.64, 1.64, 1.64,   49,   49,   49,   49,   49,   49,   49,   49,   49,   49,   49,   49, 1.64, 1.64, 1.64, 1.64]) 
motor_coil_full_length = np.array([   9,   90,   90,   90,   90,  625,  625,  625,  625,  625,  625,  625,  625,  625,  625,  625,  625,   90,   90,   90,   90]) * 1e-3 

motor_coil_od = np.zeros_like(motor_coil_node).astype(float)
motor_coil_material = []

motor_coil_length = np.array([motor_shaft_length[i] for i in motor_coil_node])
motor_coil_id = [motor_shaft_od[i] for i in motor_coil_node]

for i in range(len(motor_coil_node)):
    model_mass = motor_coil_mass[i] * motor_coil_length[i] / motor_coil_full_length[i]
    model_inertia = motor_coil_inertia[i] * model_mass / motor_coil_mass[i]
    motor_coil_od[i] = 2 * np.sqrt(2 * model_inertia / model_mass)

    model_volum = (
            np.pi
            * (motor_coil_od[i] ** 2 - motor_coil_id[i] ** 2)
            / 4
            * motor_coil_length[i]
    )

    model_rho = model_mass / model_volum
    motor_coil_material.append(
        rs.Material(name="AISI_4140", rho=model_rho, E=1, Poisson=0.3, color="cyan")
    )

motor_coil_elements = [
    rs.ShaftElement(
        L=motor_coil_length[i],
        idl=motor_coil_id[i],
        odl=motor_coil_od[i],
        material=motor_coil_material[i],
        n=motor_coil_node[i],
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(motor_coil_material))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.2.3. Modelagem dos mancais do motor
    </span>
</div>

Concluímos a modelagem do motor definindo os mancais de suporte. É importante notar que os coeficientes de rigidez e amortecimento para os mancais **DE** (*Drive End* — lado acionado) e **NDE** (*Non-Drive End* — lado oposto ao acionamento) variam conforme a velocidade de rotação do motor. 

Para que o **ROSS** considere essa dependência, devemos fornecer as propriedades em diversas velocidades distintas. Dessa forma, o pacote será capaz de realizar as interpolações necessárias para obter os valores aproximados em qualquer ponto da faixa de operação.

In [42]:
motor_bearing_de_frequency  = np.array([  47.1238898,   83.7758041, 120.42771839, 157.07963268, 193.73154697, 230.38346126, 267.03537556, 303.68728985, 340.33920414, 376.99111843])  
motor_bearing_de_kxx        = np.array([     1.13e09,      1.19e09,      1.22e09,      1.23e09,      1.25e09,      1.25e09,      1.26e09,      1.27e09,      1.27e09,      1.28e09])  
motor_bearing_de_kxz        = np.array([    -7.22e08,     -2.56e08,     -4.76e07,      8.32e07,      1.80e08,      2.60e08,      3.30e08,      3.93e08,      4.51e08,      5.07e08])  
motor_bearing_de_kzx        = np.array([    -4.27e09,     -3.73e09,     -3.52e09,     -3.41e09,     -3.34e09,     -3.29e09,     -3.26e09,     -3.24e09,     -3.22e09,     -3.21e09])  
motor_bearing_de_kzz        = np.array([     8.86e09,      5.87e09,      4.65e09,      3.96e09,      3.49e09,      3.15e09,      2.88e09,      2.65e09,      2.45e09,      2.27e09])  
motor_bearing_de_cxx        = np.array([  15400000.0,   10900000.0,    8940000.0,    7750000.0,    6890000.0,    6200000.0,    5610000.0,    5080000.0,    4600000.0,    4150000.0])  
motor_bearing_de_cxz        = np.array([ -30800000.0,  -18500000.0,  -13600000.0,  -10800000.0,   -8980000.0,   -7640000.0,   -6590000.0,   -5720000.0,   -4980000.0,   -4330000.0])  
motor_bearing_de_czx        = np.array([ -45900000.0,  -25800000.0,  -17900000.0,  -13700000.0,  -11100000.0,   -9240000.0,   -7910000.0,   -6890000.0,   -6070000.0,   -5400000.0])  
motor_bearing_de_czz        = np.array([     1.80e08,      9.33e07,      6.05e07,      4.37e07,      3.39e07,      2.78e07,      2.38e07,      2.12e07,      1.95e07,      1.84e07])  
motor_bearing_nde_frequency = np.array([  47.1238898,   83.7758041, 120.42771839, 157.07963268, 193.73154697, 230.38346126, 267.03537556, 303.68728985, 340.33920414, 376.99111843])  
motor_bearing_nde_kxx       = np.array([     1.13e09,      1.19e09,      1.22e09,      1.23e09,      1.25e09,      1.25e09,      1.26e09,      1.27e09,      1.27e09,      1.28e09])  
motor_bearing_nde_kxz       = np.array([    -7.22e08,     -2.56e08,     -4.76e07,      8.32e07,      1.80e08,      2.60e08,      3.30e08,      3.93e08,      4.51e08,      5.07e08])  
motor_bearing_nde_kzx       = np.array([    -4.27e09,     -3.73e09,     -3.52e09,     -3.41e09,     -3.34e09,     -3.29e09,     -3.26e09,     -3.24e09,     -3.22e09,     -3.21e09])  
motor_bearing_nde_kzz       = np.array([     8.86e09,      5.87e09,      4.65e09,      3.96e09,      3.49e09,      3.15e09,      2.88e09,      2.65e09,      2.45e09,      2.27e09])  
motor_bearing_nde_cxx       = np.array([  15400000.0,   10900000.0,    8940000.0,    7750000.0,    6890000.0,    6200000.0,    5610000.0,    5080000.0,    4600000.0,    4150000.0])  
motor_bearing_nde_cxz       = np.array([ -30800000.0,  -18500000.0,  -13600000.0,  -10800000.0,   -8980000.0,   -7640000.0,   -6590000.0,   -5720000.0,   -4980000.0,   -4330000.0])  
motor_bearing_nde_czx       = np.array([ -45900000.0,  -25800000.0,  -17900000.0,  -13700000.0,  -11100000.0,   -9240000.0,   -7910000.0,   -6890000.0,   -6070000.0,   -5400000.0])  
motor_bearing_nde_czz       = np.array([  15400000.0,   10900000.0,    8940000.0,    7750000.0,    6890000.0,    6200000.0,    5610000.0,    5080000.0,    4600000.0,    4150000.0])  

motor_bearing_node = [29, 3]
motor_bearing_tag = ["Motor A", "Motor B"]
motor_bearing_kxx = [motor_bearing_de_kxx, motor_bearing_nde_kxx]
motor_bearing_kyy = [motor_bearing_de_kzz, motor_bearing_nde_kzz]
motor_bearing_kxy = [motor_bearing_de_kxz, motor_bearing_nde_kxz]
motor_bearing_kyx = [motor_bearing_de_kzx, motor_bearing_nde_kzx]
motor_bearing_cxx = [motor_bearing_de_cxx, motor_bearing_nde_cxx]
motor_bearing_cyy = [motor_bearing_de_czz, motor_bearing_nde_czz]
motor_bearing_cxy = [motor_bearing_de_cxz, motor_bearing_nde_cxz]
motor_bearing_cyx = [motor_bearing_de_czx, motor_bearing_nde_czx]
motor_bearing_freq = [motor_bearing_de_frequency, motor_bearing_nde_frequency]

motor_bearing_elements = [
    rs.BearingElement(
        n=motor_bearing_node[i],
        kxx=motor_bearing_kxx[i],
        kyy=motor_bearing_kyy[i],
        kxy=motor_bearing_kxy[i],
        kyx=motor_bearing_kyx[i],
        cxx=motor_bearing_cxx[i],
        cyy=motor_bearing_cyy[i],
        cxy=motor_bearing_cxy[i],
        cyx=motor_bearing_cyx[i],
        frequency=motor_bearing_freq[i],
    )
    for i in range(len(motor_bearing_node))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.2.4. Criação do motor
    </span>
</div>

Com todos os componentes individuais definidos, o passo final consiste em integrar os elementos de eixo e mancais para instanciar a classe `Rotor`. Este objeto consolidará as propriedades físicas e geométricas, gerando a representação numérica completa do nosso motor.

In [43]:
motor_shaft_elements = motor_shaft_elements + motor_coil_elements
motor_disk_elements = [] # não há elementos de disco na modelagem, por isso a lista vazia
motor = rs.Rotor(motor_shaft_elements, motor_disk_elements, motor_bearing_elements)
motor.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.3. Modelagem do acoplamento de baixa velocidade
    </span>
</div>

Para este modelo, os acoplamentos são constituídos integralmente por elementos de eixo. 

Vale ressaltar que o **ROSS** disponibiliza classes específicas para acoplamentos (`CouplingElement`), ideais para quando as propriedades globais de rigidez e massa são conhecidas. No entanto, como dispomos apenas da geometria detalhada e das propriedades do material, utilizaremos elementos de eixo para modelá-los. Essa abordagem possibilita que o **ROSS** compute, com precisão, a distribuição de massa e inércia dos acoplamentos com base em suas dimensões físicas.

In [7]:
low_speed_coupling_material = rs.Material(
    name="low_speed_Coupling_Material", rho=7491, E=2e11, Poisson=0.3
)

low_speed_coupling_shaft_length = np.array([  32.65,   37.76,   129.6,   67.34, 105.356, 105.356, 105.356, 105.356, 105.356,    39.8,    35.0]) * 1e-3  
low_speed_coupling_shaft_id     = np.array([  109.2,   109.2,   109.2,   109.2,   109.2,   109.2,   109.2,   109.2,   109.2,   109.2,   109.2]) * 1e-3  
low_speed_coupling_shaft_od     = np.array([ 332.48,  222.34,   134.9,   205.7,   134.9,   134.9,   134.9,   134.9,   134.9,     225,   332.5]) * 1e-3  

low_speed_coupling_shaft_elements = [
    rs.ShaftElement(
        L=low_speed_coupling_shaft_length[i],
        idl=low_speed_coupling_shaft_id[i],
        odl=low_speed_coupling_shaft_od[i],
        material=low_speed_coupling_material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(low_speed_coupling_shaft_length))
]

low_speed_coupling = rs.Rotor(low_speed_coupling_shaft_elements)
low_speed_coupling.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.4. Modelagem do acoplamento de alta velocidade
    </span>
</div>

In [8]:
high_speed_coupling_material = rs.Material(
    name="High_Speed_Coupling_Material", rho=8590, E=2e11, Poisson=0.3
)

high_speed_coupling_shaft_length = np.array([ 24.0,  14.0, 94.38, 94.38, 94.38, 94.38, 94.38,  14.0,  24.0]) * 1e-3 
high_speed_coupling_shaft_id     = np.array([ 43.7,  43.7,  43.7,  43.7,  43.7,  43.7,  43.7,  43.7,  43.7]) * 1e-3 
high_speed_coupling_shaft_od     = np.array([177.8, 129.5,  54.5,  54.5,  54.5,  54.5,  54.5, 129.6, 177.8]) * 1e-3 

high_speed_coupling_shaft_elements = [
    rs.ShaftElement(
        L=high_speed_coupling_shaft_length[i],
        idl=high_speed_coupling_shaft_id[i],
        odl=high_speed_coupling_shaft_od[i],
        material=high_speed_coupling_material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(high_speed_coupling_shaft_length))
]

high_speed_coupling = rs.Rotor(high_speed_coupling_shaft_elements)
high_speed_coupling.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.5. Modelagem do compressor
    </span>
</div>

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.5.1. Modelagem do eixo do compressor
    </span>
</div>

Iniciamos a construção do modelo do compressor definindo a geometria do seu eixo. Para isso, especificamos os elementos que o compõem, seguindo o mesmo procedimento de discretização adotado anteriormente para o motor.

In [9]:
compressor_shaft_material = rs.Material(name="Steel", rho=7850, E=2e11, Poisson=0.3)

compressor_shaft_length = np.array([ 21.0,  40.9,  43.1,  10.0,  27.0,   6.0,  31.0,  34.0,   3.0,  11.5,   2.0,
                                      6.5,   3.0,  12.0,  10.0,  55.0,  28.5,  24.5,   3.0,  25.0,  30.5,  30.5,
                                    24.84, 19.16,  31.5,  31.5, 24.85, 19.15,  32.0,  32.0, 24.82, 19.18,  32.0,
                                     32.0, 25.39, 18.61,  32.0,  32.0, 25.38, 18.62,  32.5,  32.5, 25.32, 18.68,
                                     32.5,  32.5, 25.37, 18.63,   7.0,   7.0,  23.5,   3.0,  24.5,  40.5,  37.0,
                                     67.0,  10.0,  12.0,   5.0,   5.5,   2.0,  11.5,   3.0,  31.5,  34.0, 66.25,
                                      6.0,  25.0,  13.0,  25.0]) * 1e-3 

compressor_shaft_od = np.array([ 70.0,  70.0,  70.0,  70.0,  72.0,  72.0,  80.0,  80.0,  78.0,  82.0,  80.0,  82.0,
                                 80.0,  86.0,  84.0,  93.0,  95.0, 101.0,  96.2, 125.0, 146.0, 146.0, 146.0, 146.0,
                                146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0,
                                146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0,
                                146.0, 146.0, 125.0, 101.0,  96.2,  95.0,  94.0,  93.0,  84.0,  86.0,  80.0,  82.0,
                                 80.0,  82.0,  78.0,  80.0,  80.0,  54.0,  45.0,  45.0,  45.0,
                                 32.0]) * 1e-3 

compressor_shaft_id = np.zeros_like(compressor_shaft_od)

compressor_shaft_elements = [
    rs.ShaftElement(
        L=compressor_shaft_length[i],
        idl=compressor_shaft_id[i],
        odl=compressor_shaft_od[i],
        material=compressor_shaft_material,
        n=i,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(compressor_shaft_length))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.5.2. Modelagem dos selos
    </span>
</div>

Para representar fielmente os selos, utilizaremos uma combinação de dois tipos de elementos:
* **Elementos de Selo (`SealElement`):** Responsáveis pela introdução dos coeficientes de rigidez e amortecimento.
* **Elementos de Eixo (`ShaftElement`):** Utilizados para contabilizar exclusivamente a massa física dos selos. Para que esses elementos não interfiram na rigidez estrutural do rotor, utilizaremos um material "fictício" com módulo de elasticidade ($E$) desprezível (próximo de zero).

Essa abordagem híbrida garante que o modelo final considere não apenas as forças de reação dos selos, mas também sua influência na distribuição de massa e, consequentemente, nas frequências naturais do sistema.

Primeiramente, definiremos os elementos de eixo responsáveis por representar a massa física dos selos.

In [10]:
compressor_seal_shaft_material = rs.Material(
    name="Steel", rho=9084.5, E=1, Poisson=0.3, color="red"
)

compressor_seal_shaft_length = np.array([ 21.0,  40.9,  43.1,  10.0,  27.0,   6.0,  31.0,  34.0,   3.0,  11.5,   2.0,
                                           6.5,   3.0,  12.0,  10.0,  55.0,  28.5,  24.5,   3.0,  25.0,  30.5,  30.5,
                                         24.84, 19.16,  31.5,  31.5, 24.85, 19.15,  32.0,  32.0, 24.82, 19.18,  32.0,
                                          32.0, 25.39, 18.61,  32.0,  32.0, 25.38, 18.62,  32.5,  32.5, 25.32, 18.68,
                                          32.5,  32.5, 25.37, 18.63,   7.0,   7.0,  23.5,   3.0,  24.5,  40.5,  37.0,
                                          67.0,  10.0,  12.0,   5.0,   5.5,   2.0,  11.5,   3.0,  31.5,  34.0, 66.25,
                                           6.0,  25.0,  13.0,  25.0]) * 1e-3  

compressor_seal_shaft_od = np.array([104.9,  70.0,  70.0,  70.0,  78.0,  78.0,  80.0,  80.0,  78.0,  82.0,  80.0,  82.0,
                                      80.0,  92.0, 128.0, 164.0, 164.0, 101.0, 101.0, 158.0, 228.0, 228.0, 146.0, 146.0,
                                     170.0, 170.0, 146.0, 146.0, 170.0, 170.0, 146.0, 146.0, 170.0, 170.0, 146.0, 146.0,
                                     170.0, 170.0, 146.0, 146.0, 170.0, 170.0, 146.0, 146.0, 170.0, 170.0, 146.0, 146.0,
                                     170.0, 170.0, 156.0, 101.0, 101.0, 164.0, 164.0, 164.0, 128.0,  92.0,  80.0,  82.0,
                                      80.0,  82.0,  78.0,  80.0,  80.0,  60.1,  60.0, 139.0,  60.0,
                                      60.0]) * 1e-3 + 1e-10  

compressor_seal_shaft_id = np.array([ 70.0,  70.0,  70.0,  70.0,  72.0,  72.0,  80.0,  80.0,  78.0,  82.0,  80.0,  82.0,
                                      80.0,  86.0,  84.0,  93.0,  95.0, 101.0,  96.2, 125.0, 146.0, 146.0, 146.0, 146.0,
                                     146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0,
                                     146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0,
                                     146.0, 146.0, 125.0, 101.0,  96.2,  95.0,  94.0,  93.0,  84.0,  86.0,  80.0,  82.0,
                                      80.0,  82.0,  78.0,  80.0,  80.0,  54.0,  45.0,  45.0,  45.0,
                                      32.0]) * 1e-3 

compressor_seal_shaft_elements = [
    rs.ShaftElement(
        L=compressor_seal_shaft_length[i],
        idl=compressor_seal_shaft_id[i],
        odl=compressor_seal_shaft_od[i],
        material=compressor_seal_shaft_material,
        n=i,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(compressor_seal_shaft_length))
]

Após representarmos a massa física, passamos para a modelagem dos **efeitos dinâmicos** dos selos. Para isso, utilizaremos a classe `SealElement`, que é responsável por introduzir os coeficientes de rigidez e amortecimento resultantes da interação entre o fluido e as vedações do compressor.

Como o projeto contempla oito selos distintos, as propriedades foram estruturadas em listas. 

In [11]:
compressor_seal_node     = np.array([      21,       25,       29,       33,       37,       41,       45,       49])  
compressor_seal_node_kxx = np.array([-1167400,   -72507,   -71340,   -70169,   -68436,   -67273,   -66110,   -64947]).astype(float)  
compressor_seal_node_kyy = np.array([-1167400,   -72507,   -71340,   -70169,   -68436,   -67273,   -66110,   -64947]).astype(float)  
compressor_seal_node_kxy = np.array([  -90020,   -40239,   -39629,   -38983,   -38533,   -37878,   -37223,   -36566]).astype(float)  
compressor_seal_node_kyx = np.array([   90020,    40239,    39629,    38983,    38533,    37878,    37223,    36566]).astype(float)  
compressor_seal_node_cxx = np.array([  773.14,   134.54,   131.78,   129.73,   129.42,   127.34,   125.26,   123.19])  
compressor_seal_node_cyy = np.array([  773.14,   134.54,   131.78,   129.73,   129.42,   127.34,   125.26,   123.19])  
compressor_seal_node_cxy = np.array([  757.95,    51.06,    50.07,    49.28,    48.78,    47.99,     47.2,    46.41])  
compressor_seal_node_cyx = np.array([ -757.95,   -51.06,   -50.07,   -49.28,   -48.78,   -47.99,    -47.2,   -46.41])  

compressor_seal_elements = [
    rs.SealElement(
        n=compressor_seal_node[i],
        kxx=compressor_seal_node_kxx[i],
        kyy=compressor_seal_node_kyy[i],
        kxy=compressor_seal_node_kxy[i],
        kyx=compressor_seal_node_kyx[i],
        cxx=compressor_seal_node_cxx[i],
        cyy=compressor_seal_node_cyy[i],
        cxy=compressor_seal_node_cxy[i],
        cyx=compressor_seal_node_cyx[i],
    )
    for i in range(len(compressor_seal_node))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.5.3. Modelagem dos discos do compressor
    </span>
</div>

Diferente do motor, que apresentava uma estrutura mais simplificada, o compressor conta com diversos discos - que representam os **impelidores** de cada estágio. Para incorporá-los ao projeto, utilizaremos a classe `DiskElement`, especificando as massas, inércias, e seus respectivos nós de fixação.

In [12]:
compressor_disk_node          = np.array([    23,     27,     31,     35,     39,     43,     47])  
compressor_disk_mass          = np.array([ 10.93,  10.91,  10.87,  11.14,   11.1,  11.05,  11.01])  
compressor_disk_inertia       = np.array([0.0916, 0.0915, 0.0912, 0.0933, 0.0931, 0.0928, 0.0925])  
compressor_disk_polar_inertia = np.array([0.1792, 0.1789, 0.1784, 0.1824, 0.1819, 0.1814, 0.1808])  

compressor_disk_elements = [
    rs.DiskElement(
        n=compressor_disk_node[i],
        m=compressor_disk_mass[i],
        Id=compressor_disk_inertia[i],
        Ip=compressor_disk_polar_inertia[i],
    )
    for i in range(len(compressor_disk_node))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.5.4. Modelagem dos mancais do compressor
    </span>
</div>

Para concluir a definição dos componentes do compressor, modelaremos os mancais de suporte **DE** e **NDE**. Assim como observado no motor, as propriedades de rigidez e amortecimento destes mancais não são estáticas; elas variam significativamente de acordo com a velocidade de rotação do rotor.

Para capturar esse comportamento, forneceremos ao modelo tabelas de coeficientes em função da frequência. Isso possibilita que o **ROSS** realize a interpolação dos valores dos coeficientes, garantindo precisão tanto em baixas rotações quanto na velocidade nominal de operação.

In [13]:
compressor_bearing_de_frequency  = np.array([     1000.0,      2000.0,      3000.0,      4000.0,      5000.0,      6000.0,      7000.0,      8000.0,      9000.0,      9167.0,     10000.0,     11000.0,     12000.0,     12223.0,     12835.0,     13000.0,     14000.0,     15000.0,     16000.0,     17000.0,     18000.0,     19000.0,     20000.0]) * np.pi / 30  
compressor_bearing_de_kxx        = np.array([  1.6954e07,   3.0118e07,   4.1684e07,    5.241e07,   6.2652e07,   7.2645e07,   8.2506e07,   9.2308e07,   1.0209e08,   1.0372e08,   1.1188e08,   1.2168e08,    1.315e08,   1.3369e08,   1.3971e08,   1.4134e08,    1.512e08,   1.6108e08,   1.7098e08,   1.8088e08,   1.9081e08,   2.0074e08,   2.1069e08])                  
compressor_bearing_de_kxz        = np.array([    -3433.7,      -13793,      -31146,      -55519,      -86912,     -125370,     -170910,     -223490,     -283190,     -298840,     -350010,     -423920,     -505000,     -524070,     -578190,     -593240,     -688610,     -791210,     -901020,    -1.018e6,   -1.1423e6,   -1.2738e6,   -1.4125e6])                  
compressor_bearing_de_kzx        = np.array([     3415.1,       13683,       30939,       55232,       86569,      125000,      170520,      223100,      282820,      293480,      349660,      423600,      504720,      523800,      577950,      593000,      688410,      791070,      900930,     1.018e6,    1.1423e6,    1.2739e6,    1.4127e6])                  
compressor_bearing_de_kzz        = np.array([  2.1269e08,   1.5453e08,   1.3374e08,    1.252e08,   1.2232e08,   1.2295e08,   1.2579e08,   1.3016e08,   1.3564e08,   1.3664e08,   1.4194e08,   1.4889e08,   1.5632e08,   1.5803e08,   1.6283e08,   1.6414e08,   1.7228e08,   1.8067e08,   1.8927e08,   1.9799e08,    2.069e08,   2.1593e08,   2.2506e08])                  
compressor_bearing_de_cxx        = np.array([     109560,       99414,       93128,       88831,       85722,       83426,       81686,       80347,       79300,       79148,       78469,       77804,       77261,       77154,       76884,       76817,       76448,       76140,       75880,       75655,       75465,       75303,       75162])                  
compressor_bearing_de_cxz        = np.array([     11.825,      22.969,      34.111,      45.269,      56.443,      67.666,      78.929,      90.221,      101.57,      103.46,      112.96,      124.38,      135.85,      138.42,      145.47,      147.37,      158.92,      170.53,      182.18,      193.85,      205.58,      217.36,      229.17])                  
compressor_bearing_de_czx        = np.array([    -11.008,     -22.083,     -33.277,     -44.531,     -55.813,     -67.141,       -78.5,     -89.879,      -101.3,     -103.21,     -112.76,     -124.24,     -135.77,     -138.35,     -145.43,     -147.34,     -158.93,     -170.57,     -182.25,     -193.96,     -205.72,     -217.52,     -229.35])                  
compressor_bearing_de_czz        = np.array([     794140,      361240,      232630,      175290,      144080,      125260,      112990,      104570,       98543,       97707,       94097,       90732,       88117,       87615,       86365,       86056,       84402,       83055,       81946,       81006,       80227,       79566,       79001])                  
compressor_bearing_nde_frequency = np.array([       1000,        2000,        3000,        4000,        5000,        6000,        7000,        8000,        9000,        9167,       10000,       11000,       12000,       12223,       12835,       13000,       14000,       15000,       16000,       17000,       18000,       19000,       20000]) * np.pi / 30  
compressor_bearing_nde_kxx       = np.array([  1.6671e07,   2.9483e07,   4.0758e07,    5.125e07,   6.1368e07,   7.1261e07,   8.1079e07,    9.087e07,   1.0066e08,    1.023e08,   1.1048e08,   1.2031e08,   1.3017e08,   1.3237e08,   1.3842e08,   1.4006e08,   1.4996e08,   1.5988e08,   1.6981e08,   1.7976e08,   1.8973e08,    1.997e08,   2.0968e08])                  
compressor_bearing_nde_kxz       = np.array([    -3435.4,      -13802,      -31166,      -55551,      -86955,     -125430,     -170970,     -223560,     -283270,     -293920,     -350090,     -423990,     -505070,     -524140,     -578270,     -593320,     -686860,     -791280,     -901090,  -1.0181e06,  -1.1423e06,  -1.2739e06,  -1.4126e06])                  
compressor_bearing_nde_kzx       = np.array([     3413.2,       13693,       30970,       55288,       86648,      125100,      170630,      223220,      282950,      293600,      349790,      423730,      504840,      523920,      578060,      593120,      688520,      791160,      901020,    1.018e06,   1.1423e06,   1.2739e06,   1.4127e06])                  
compressor_bearing_nde_kzz       = np.array([  1.8037e08,    1.324e08,   1.1625e08,   1.1038e08,   1.0968e08,   1.1173e08,   1.1577e08,   1.2114e08,   1.2745e08,   1.2857e08,   1.3446e08,   1.4199e08,   1.4994e08,   1.5176e08,   1.5683e08,   1.5821e08,   1.6674e08,   1.7548e08,   1.8433e08,   1.9337e08,   2.0253e08,   2.1178e08,   2.2112e08])                  
compressor_bearing_nde_cxx       = np.array([     108040,       97698,       91446,       87250,       84310,       82155,       80563,       79356,       78426,       78292,       77697,       77118,       76652,       76561,       76330,       76273,       75960,       75700,       75477,       75293,       75135,       75001,       74885])                  
compressor_bearing_nde_cxz       = np.array([     11.754,      22.881,      34.011,      45.161,      56.332,      67.552,      78.815,      90.109,      101.46,      103.35,      112.85,      124.27,      135.75,      138.32,      145.37,      147.27,      158.82,      170.43,      182.08,      193.76,      205.49,      217.27,      229.08])                  
compressor_bearing_nde_czx       = np.array([        -11,     -22.083,     -33.277,     -44.523,     -55.796,     -67.112,      -78.46,     -89.829,     -101.24,     -103.15,     -112.69,     -124.17,     -135.69,     -138.26,     -145.34,     -147.25,     -158.83,     -170.47,     -182.14,     -193.85,      -205.6,      -217.4,     -229.23])                  
compressor_bearing_nde_czz       = np.array([     701300,      319300,      207910,      158540,      132160,      116170,      105890,       98880,       93897,       93208,       90237,       87475,       85342,       84932,       83914,       83663,       82318,       81226,       80311,       79663,       78935,       78402,       77947])                  

compressor_bearing_node = [7, 64]
compressor_bearing_kxx = [compressor_bearing_de_kxx, compressor_bearing_nde_kxx]
compressor_bearing_kyy = [compressor_bearing_de_kzz, compressor_bearing_nde_kzz]
compressor_bearing_kxy = [compressor_bearing_de_kxz, compressor_bearing_nde_kxz]
compressor_bearing_kyx = [compressor_bearing_de_kzx, compressor_bearing_nde_kzx]
compressor_bearing_cxx = [compressor_bearing_de_cxx, compressor_bearing_nde_cxx]
compressor_bearing_cyy = [compressor_bearing_de_czz, compressor_bearing_nde_czz]
compressor_bearing_cxy = [compressor_bearing_de_cxz, compressor_bearing_nde_cxz]
compressor_bearing_cyx = [compressor_bearing_de_czx, compressor_bearing_nde_czx]
compressor_bearing_frequency = [
    compressor_bearing_de_frequency,
    compressor_bearing_nde_frequency,
]

compressor_bearing_elements = [
    rs.BearingElement(
        n=compressor_bearing_node[i],
        kxx=compressor_bearing_kxx[i],
        kyy=compressor_bearing_kyy[i],
        kxy=compressor_bearing_kxy[i],
        kyx=compressor_bearing_kyx[i],
        cxx=compressor_bearing_cxx[i],
        cyy=compressor_bearing_cyy[i],
        cxy=compressor_bearing_cxy[i],
        cyx=compressor_bearing_cyx[i],
        frequency=compressor_bearing_frequency[i],
    )
    for i in range(len(compressor_bearing_node))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        1.5.5. Criação do compressor
    </span>
</div>

Agora que todos os componentes do compressor foram devidamente especificados (incluindo a geometria do eixo, os discos, os mancais e a estratégia híbrida para os selos) o próximo passo é criar o objeto `Rotor`. 

Esta etapa consolida os elementos individuais em um modelo matemático unificado. A partir daqui, seremos capazes de visualizar a montagem completa e validar se a distribuição de componentes e nós está correta antes de avançarmos para a caixa de engrenagens.

In [14]:
compressor_bearing_elements.extend(compressor_seal_elements)
compressor_shaft_elements = compressor_shaft_elements + compressor_seal_shaft_elements
compressor = rs.Rotor(
    compressor_shaft_elements,
    compressor_disk_elements,
    compressor_bearing_elements,
)
compressor.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 32px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2. Modelagem da caixa de engrenagens
    </span>
</div>

<p align="center"><img src="figs/caixa_engrenagens.png" width="65%"></p>

Nas etapas anteriores (motor, compressor e acoplamentos), utilizamos ferramentas fundamentais já exploradas no curso, como elementos de eixo, discos, mancais e selos. Contudo, para conectar dois rotores distintos e consolidar um **sistema MultiRotor**, precisamos introduzir um novo componente: o elemento de engrenagem, `GearElement` ou `GearElementTVMS`.

Nesta seção, veremos que existem diferentes abordagens para definir esse elemento no **ROSS** e compreenderemos como ele atua na transferência de esforços e no acomplamento das dinâmicas de cada rotor para a construção do modelo final.

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.1. Modelagem do engrenamento de baixa velocidade
    </span>
</div>

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.1.1. Modelagem do eixo do engrenamento de baixa velocidade
    </span>
</div>

É fundamental considerar que o subsistema de engrenamento não consiste apenas na engrenagem em si; ele abrange também o eixo e os mancais que garantem a sustentação e o posicionamento do conjunto. 

Iniciaremos a modelagem definindo esses elementos de suporte. Como você notará no código a seguir, seguiremos o padrão de discretização e parametrização de materiais com o qual já estamos familiarizados desde o início do curso.

In [15]:
low_speed_gearbox_shaft_material = rs.Material(
    name="AISI_4140", rho=7794.17, E=2e11, Poisson=0.3
)

low_speed_gearbox_shaft_length = np.array([0.08557457, 0.00244499, 0.02322738, 0.09168704, 0.09413203, 0.01100244,
                                           0.03300733, 0.02689487, 0.04767726, 0.04767726, 0.02200489, 0.03789731,
                                           0.0391198, 0.02444988, 0.02444988, 0.03789731,  0.0391198,  0.0207824,
                                           0.04767726, 0.04645477, 0.02811736,  0.0403423,  0.0599022, 0.06356968,
                                           0.03056235])  

low_speed_gearbox_shaft_od = np.array([0.19518072, 0.29879518, 0.38795181, 0.19277108, 0.19036145, 0.21686747, 0.19518072,
                                       0.19277108, 0.19036145, 0.19036145, 0.19036145, 0.78072289, 0.78072289, 0.76144578,
                                       0.76144578, 0.78072289, 0.78072289, 0.19036145, 0.19036145, 0.19036145, 0.19036145,
                                       0.19036145, 0.08433735, 0.08433735, 0.18072289])  

low_speed_gearbox_shaft_id = np.zeros_like(low_speed_gearbox_shaft_od)
low_speed_gearbox_shaft_elements = [
    rs.ShaftElement(
        L=low_speed_gearbox_shaft_length[i],
        idl=low_speed_gearbox_shaft_id[i],
        odl=low_speed_gearbox_shaft_od[i],
        material=low_speed_gearbox_shaft_material,
    )
    for i in range(len(low_speed_gearbox_shaft_length))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.1.2. Modelagem dos mancais do engrenamento de baixa velocidade
    </span>
</div>

In [16]:
low_speed_gearbox_bearing_freq      = np.array([     410,      820,     1231,     1641,     2051,     2461,     2871,     3282]) * np.pi/30  
low_speed_gearbox_bearing_blind_kxx = np.array([ 1.26e08,  1.41e08,  1.69e08,  1.82e08,  1.99e08,  2.16e08,  2.30e08,  2.41e08])  
low_speed_gearbox_bearing_blind_kxy = np.array([ 4.33e08,  3.27e08,  3.00e08,  2.85e08,  2.81e08,  2.82e08,  2.85e08,  2.86e08])  
low_speed_gearbox_bearing_blind_kyx = np.array([-4.98e08, -5.77e08, -6.53e08, -6.86e08, -7.25e08, -7.62e08, -7.95e08, -8.22e08])  
low_speed_gearbox_bearing_blind_kyy = np.array([ 1.90e09,  1.40e09,  1.20e09,  1.10e09,  1.05e09,  1.01e09,  9.87e08,  9.78e08])  
low_speed_gearbox_bearing_blind_cxx = np.array([ 3.04e06,  1.93e06,  1.59e06,  1.37e06,  1.25e06,  1.18e06,  1.11e06,  1.04e06])  
low_speed_gearbox_bearing_blind_cxy = np.array([ 4.22e06,  9.47e05,  2.20e05, -9.41e04, -2.24e05, -2.99e05, -3.35e05, -3.51e05])  
low_speed_gearbox_bearing_blind_cyx = np.array([ 8.52e05, -4.68e05, -6.70e05, -6.04e05, -5.79e05, -5.56e05, -5.25e05, -4.77e05])  
low_speed_gearbox_bearing_blind_cyy = np.array([ 5.13e07,  2.30e07,  1.48e07,  1.11e07,  8.88e06,  7.49e06,  6.49e06,  5.75e06])  
low_speed_gearbox_bearing_ext_kxx   = np.array([ 1.70e08,  1.93e08,  2.03e08,  2.24e08,  2.43e08,  2.62e08,  2.81e08,  2.96e08])  
low_speed_gearbox_bearing_ext_kxy   = np.array([ 4.37e08,  3.32e08,  2.93e08,  2.81e08,  2.76e08,  2.80e08,  2.84e08,  2.88e08])  
low_speed_gearbox_bearing_ext_kyx   = np.array([-7.33e08, -7.69e08, -7.75e08, -8.14e08, -8.47e08, -8.82e08, -9.16e08, -9.44e08])  
low_speed_gearbox_bearing_ext_kyy   = np.array([ 2.38e09,  1.72e09,  1.48e09,  1.34e09,  1.26e09,  1.20e09,  1.17e09,  1.14e09])  
low_speed_gearbox_bearing_ext_cxx   = np.array([ 3.27e06,  2.20e06,  1.72e06,  1.52e06,  1.37e06,  1.27e06,  1.21e06,  1.15e06])  
low_speed_gearbox_bearing_ext_cxy   = np.array([ 2.71e06,  1.18e06, -4.18e05, -5.37e05, -5.75e05, -5.83e05, -5.92e05, -5.77e05])  
low_speed_gearbox_bearing_ext_cyx   = np.array([-1.53e06, -1.71e06, -1.24e06, -1.09e06, -9.43e05, -8.49e05, -7.92e05, -7.23e05])  
low_speed_gearbox_bearing_ext_cyy   = np.array([ 6.18e07,  2.73e07,  1.74e07,  1.28e07,  1.02e07,  8.48e06,  7.33e06,  6.45e06])  

low_speed_gearbox_bearing_node = [9, 19]
low_speed_gearbox_bearing_tag = ["Gearbox A", "Gearbox B"]
low_speed_gearbox_bearing_kxx = [
    low_speed_gearbox_bearing_ext_kxx,
    low_speed_gearbox_bearing_blind_kxx,
]
low_speed_gearbox_bearing_kyy = [
    low_speed_gearbox_bearing_ext_kyy,
    low_speed_gearbox_bearing_blind_kyy,
]
low_speed_gearbox_bearing_kxy = [
    low_speed_gearbox_bearing_ext_kxy,
    low_speed_gearbox_bearing_blind_kxy,
]
low_speed_gearbox_bearing_kyx = [
    low_speed_gearbox_bearing_ext_kyx,
    low_speed_gearbox_bearing_blind_kyx,
]
low_speed_gearbox_bearing_cxx = [
    low_speed_gearbox_bearing_ext_cxx,
    low_speed_gearbox_bearing_blind_cxx,
]
low_speed_gearbox_bearing_cyy = [
    low_speed_gearbox_bearing_ext_cyy,
    low_speed_gearbox_bearing_blind_cyy,
]
low_speed_gearbox_bearing_cxy = [
    low_speed_gearbox_bearing_ext_cxy,
    low_speed_gearbox_bearing_blind_cxy,
]
low_speed_gearbox_bearing_cyx = [
    low_speed_gearbox_bearing_ext_cyx,
    low_speed_gearbox_bearing_blind_cyx,
]

low_speed_gearbox_bearing_elements = [
    rs.BearingElement(
        frequency=low_speed_gearbox_bearing_freq,
        n=low_speed_gearbox_bearing_node[i],
        kxx=low_speed_gearbox_bearing_kxx[i],
        kyy=low_speed_gearbox_bearing_kyy[i],
        kxy=low_speed_gearbox_bearing_kxy[i],
        kyx=low_speed_gearbox_bearing_kyx[i],
        cxx=low_speed_gearbox_bearing_cxx[i],
        cyy=low_speed_gearbox_bearing_cyy[i],
        cxy=low_speed_gearbox_bearing_cxy[i],
        cyx=low_speed_gearbox_bearing_cyx[i],
        tag=low_speed_gearbox_bearing_tag[i],
    )
    for i in range(len(low_speed_gearbox_bearing_node))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.1.3. Criação do engrenamento de baixa velocidade (sem engrenagem)
    </span>
</div>

In [17]:
low_speed_gearbox = rs.Rotor(
    low_speed_gearbox_shaft_elements,
    [],
    low_speed_gearbox_bearing_elements,
)
low_speed_gearbox.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.2. Modelagem do engrenamento de alta velocidade
    </span>
</div>

Para modelar o engrenamento de alta velocidade, seguiremos um procedimento análogo ao realizado para o engrenamento de baixa velocidade. O fluxo de trabalho consiste na definição sequencial dos elementos que compõem o eixo, seguida pela configuração de seus respectivos mancais e, para concluir, pela especificação da engrenagem que integra este conjunto.

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.2.1. Modelagem do eixo do engrenamento de alta velocidade
    </span>
</div>

In [1]:
high_speed_gearbox_shaft_material = rs.Material(
    name="AISI_4140", rho=7975.6, E=2e11, Poisson=0.3
)

high_speed_gearbox_shaft_length = np.array([0.00887097, 0.01209677, 0.04112903, 0.03064516, 0.01532258, 0.03145161,
                                            0.03306452, 0.00967742, 0.01451613,      0.025, 0.04112903, 0.03870968,
                                            0.02903226, 0.01048387, 0.03870968, 0.03870968,  0.0233871, 0.02419355,
                                            0.03870968, 0.03951613, 0.01048387, 0.01935484, 0.03870968, 0.03870968,
                                            0.03629032, 0.03387097, 0.02580645, 0.02822581, 0.03870968])  

high_speed_gearbox_shaft_od = np.array([0.08032787, 0.08032787, 0.05737705, 0.05737705,
                                        0.05737705, 0.07868852, 0.07868852, 0.10655738,
                                        0.07868852, 0.07868852, 0.08032787, 0.07868852,
                                        0.08032787, 0.10655738, 0.12459016, 0.12459016,
                                        0.10655738, 0.10819672, 0.12459016, 0.12459016,
                                        0.10655738, 0.08032787, 0.08032787, 0.08032787,
                                        0.08032787, 0.08032787, 0.07377049, 0.07377049,
                                        0.0899])  # [m] 

high_speed_gearbox_shaft_id = np.zeros_like(high_speed_gearbox_shaft_length)

high_speed_gearbox_shaft_elements = [
    rs.ShaftElement(
        L=high_speed_gearbox_shaft_length[i],
        idl=high_speed_gearbox_shaft_id[i],
        odl=high_speed_gearbox_shaft_od[i],
        material=high_speed_gearbox_shaft_material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(high_speed_gearbox_shaft_length))
]

NameError: name 'rs' is not defined

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.2.2. Modelagem dos mancais do engrenamento de alta velocidade
    </span>
</div>

In [19]:
high_speed_gearbox_bearing_freq       = np.array([    2567,     5134,     7702,    10269,    12836,    15403,    17970,    20538]) * np.pi/30  
high_speed_gearbox_bearing_blind_kxx  = np.array([ 8.16e08,  6.55e08,  6.24e08,  6.22e08,  6.29e08,  6.40e08,  6.54e08,  6.68e08])  
high_speed_gearbox_bearing_blind_kyx  = np.array([-1.10e09, -8.37e08, -7.70e08, -7.52e08, -7.53e08, -7.64e08, -7.79e08, -7.96e08])  
high_speed_gearbox_bearing_blind_kxy  = np.array([-2.75e08, -5.44e07,  4.11e07,  8.72e07,  1.15e08,  1.27e08,  1.32e08,  1.35e08])  
high_speed_gearbox_bearing_blind_kyy  = np.array([ 7.92e08,  4.72e08,  3.70e08,  3.27e08,  3.05e08,  2.92e08,  2.83e08,  2.77e08])  
high_speed_gearbox_bearing_blind_cxx  = np.array([ 3.52e06,  1.65e06,  1.12e06,  8.53e05,  6.94e05,  5.77e05,  4.92e05,  4.29e05])  
high_speed_gearbox_bearing_blind_cxy  = np.array([-3.37e06, -1.41e06, -9.05e05, -6.65e05, -5.27e05, -4.35e05, -3.70e05, -3.22e05])  
high_speed_gearbox_bearing_blind_cyx  = np.array([-3.95e06, -1.61e06, -9.80e05, -6.94e05, -5.35e05, -4.36e05, -3.69e05, -3.21e05])  
high_speed_gearbox_bearing_blind_cyy  = np.array([ 4.77e06,  1.98e06,  1.26e06,  9.35e05,  7.54e05,  6.39e05,  5.58e05,  4.99e05])  
high_speed_gearbox_bearing_ext_kxx    = np.array([ 7.97e08,  6.49e08,  6.17e08,  6.14e08,  6.21e08,  6.33e08,  6.46e08,  6.61e08])  
high_speed_gearbox_bearing_ext_kxy    = np.array([-2.72e08, -3.35e07,  4.72e07,  8.99e07,  1.13e08,  1.23e08,  1.26e08,  1.29e08])  
high_speed_gearbox_bearing_ext_kyx    = np.array([-1.06e09, -8.12e08, -7.49e08, -7.33e08, -7.35e08, -7.46e08, -7.62e08, -7.81e08])  
high_speed_gearbox_bearing_ext_kyy    = np.array([ 7.57e08,  4.43e08,  3.53e08,  3.14e08,  2.94e08,  2.82e08,  2.74e08,  2.69e08])  
high_speed_gearbox_bearing_ext_cxx    = np.array([ 3.39e06,  1.67e06,  1.11e06,  8.42e05,  6.76e05,  5.62e05,  4.78e05,  4.18e05])  
high_speed_gearbox_bearing_ext_cxy    = np.array([-3.20e06, -1.41e06, -8.83e05, -6.49e05, -5.11e05, -4.22e05, -3.59e05, -3.13e05])  
high_speed_gearbox_bearing_ext_cyx    = np.array([-3.78e06, -1.58e06, -9.49e05, -6.71e05, -5.18e05, -4.22e05, -3.58e05, -3.12e05])  
high_speed_gearbox_bearing_ext_cyy    = np.array([ 4.53e06,  1.92e06,  1.21e06,  9.05e05,  7.31e05,  6.21e05,  5.43e05,  4.86e05])  

high_speed_gearbox_bearing_node = [11, 23]
high_speed_gearbox_bearing_kxx = [
    high_speed_gearbox_bearing_ext_kxx,
    high_speed_gearbox_bearing_blind_kxx,
]
high_speed_gearbox_bearing_kyy = [
    high_speed_gearbox_bearing_ext_kyy,
    high_speed_gearbox_bearing_blind_kyy,
]
high_speed_gearbox_bearing_kxy = [
    high_speed_gearbox_bearing_ext_kxy,
    high_speed_gearbox_bearing_blind_kxy,
]
high_speed_gearbox_bearing_kyx = [
    high_speed_gearbox_bearing_ext_kyx,
    high_speed_gearbox_bearing_blind_kyx,
]
high_speed_gearbox_bearing_cxx = [
    high_speed_gearbox_bearing_ext_cxx,
    high_speed_gearbox_bearing_blind_cxx,
]
high_speed_gearbox_bearing_cyy = [
    high_speed_gearbox_bearing_ext_cyy,
    high_speed_gearbox_bearing_blind_cyy,
]
high_speed_gearbox_bearing_cxy = [
    high_speed_gearbox_bearing_ext_cxy,
    high_speed_gearbox_bearing_blind_cxy,
]
high_speed_gearbox_bearing_cyx = [
    high_speed_gearbox_bearing_ext_cyx,
    high_speed_gearbox_bearing_blind_cyx,
]

high_speed_gearbox_bearing_elements = [
    rs.BearingElement(
        frequency=high_speed_gearbox_bearing_freq,
        n=high_speed_gearbox_bearing_node[i],
        kxx=high_speed_gearbox_bearing_kxx[i],
        kyy=high_speed_gearbox_bearing_kyy[i],
        kxy=high_speed_gearbox_bearing_kxy[i],
        kyx=high_speed_gearbox_bearing_kyx[i],
        cxx=high_speed_gearbox_bearing_cxx[i],
        cyy=high_speed_gearbox_bearing_cyy[i],
        cxy=high_speed_gearbox_bearing_cxy[i],
        cyx=high_speed_gearbox_bearing_cyx[i],
    )
    for i in range(len(high_speed_gearbox_bearing_node))
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.2.3. Criação do engrenamento de alta velocidade (sem engrenagem)
    </span>
</div>

In [20]:
high_speed_gearbox = rs.Rotor(
    high_speed_gearbox_shaft_elements,
    [],
    high_speed_gearbox_bearing_elements,
)
high_speed_gearbox.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.3. Inclusão das engrenagens
    </span>
</div>

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.3.1. Definição de variáveis comuns às engrenagens
    </span>
</div>

A construção de elementos de engrenagem no **ROSS** baseia-se na definição de parâmetros geométricos e mecânicos fundamentais. Nesta etapa, definiremos variáveis globais que serão aplicadas a ambas as engrenagens, simplificando a estrutura do código e garantindo a consistência do par de engrenamento.

Os parâmetros principais (representados na imagem abaixo) são:

* **Material:** Define propriedades como densidade e elasticidade.
* **Ângulo de Hélice (*helix angle*):** A inclinação dos dentes em relação ao eixo da engrenagem.
* **Ângulo de Pressão (*pressure angle*):** O ângulo entre a linha de ação e a tangente aos círculos primitivos no ponto de contato.
* **Módulo (*gear module*):** Razão entre o diâmetro primitivo e o número de dentes.
* **Largura da Face (*gear width*):** O comprimento do dente medido paralelamente ao eixo da engrenagem.

Como esses valores são comuns a ambas as engrenagens, definiremos essas constantes previamente para facilitar a parametrização dos objetos a seguir.

<p align="center"><img src="figs/variaveis_engrenagens.png" width="40%"></p>

<div style="
    width: 99%;
    display: flex;
    overflow: hidden;
    text-align: justify;
    border-radius: 10px;
    align-items: stretch;
    box-sizing: border-box;
    margin: -5px auto -5px 0px;
    border: 1px solid #cccccc;
">
    <div style="
        display: flex;
        max-width: 80px;
        color: #ffffff;
        font-weight: bold;
        padding: 14px 18px;
        align-items: center;
        white-space: nowrap;
        text-align: center;
        border-top-left-radius: 10px;
        border-bottom-left-radius: 10px;
        background: linear-gradient(90deg, #005c3b, #008542);
    ">
        Atividade<br>Prática
    </div>
    <div style="flex: 1; padding: 14px 18px; text-align: justify;">
        <p>
            Complete o código que se encontra a seguir (substituindo os <code>???</code>) 
            com base nos dados presentes na página 4 do documento <i>Special Purpose Gear Units - API 613 Fifth Edition - Data Sheet - SI Units</i>. 
        </p>
    </div>
</div>

In [ ]:
gearbox_gear_material = rs.Material(name="Steel", rho=7850, E=2e11, Poisson=0.3)
gearbox_gear_helix_angle = Q_(???, "degree").to_base_units().m
gearbox_gear_pressure_angle = Q_(???, "degree").to_base_units().m
gearbox_gear_width = Q_(???, "mm").to_base_units().m
gearbox_gear_module = Q_(4.0 / np.cos(gearbox_gear_helix_angle), "mm").to_base_units().m

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px; 
        max-width: 100%;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Orientação
        </summary>
        <div style="
            padding: 20px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            display: flex;
            justify-content: center;
            overflow-x: auto;
        ">
            <img src="figs/dica_ex_1.png" style="max-width: 80%; height: auto;" alt="Datasheet">
        </div>
    </details>
</div>

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px; 
        max-width: 100%;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Ver Gabarito
        </summary>
        <div style="
            padding: 10px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            color: #333333;
            overflow-x: auto;
        ">
            <button 
                onclick="navigator.clipboard.writeText(`gearbox_gear_material = rs.Material(name='Steel', rho=7850, E=2e11, Poisson=0.3)\ngearbox_gear_helix_angle = Q_(30.9749, 'degree').to_base_units().m\ngearbox_gear_pressure_angle = Q_(20, 'degree').to_base_units().m\ngearbox_gear_width = Q_(156.01, 'mm').to_base_units().m\ngearbox_gear_module = Q_(4.665, 'mm').to_base_units().m`)"
                style="
                    margin-bottom: 15px; 
                    padding: 8px 16px; 
                    background-color: #008542; 
                    color: white; 
                    border: none; 
                    border-radius: 4px; 
                    cursor: pointer; 
                    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
                    font-weight: bold;
                    display: block;
                ">
                Copiar
            </button>

```python
gearbox_gear_material = rs.Material(name="Steel", rho=7850, E=2e11, Poisson=0.3)
gearbox_gear_helix_angle = Q_(30.9749, "degree").to_base_units().m
gearbox_gear_pressure_angle = Q_(20, "degree").to_base_units().m
gearbox_gear_width = Q_(156.01, "mm").to_base_units().m
gearbox_gear_module = Q_(4.0 / np.cos(gearbox_gear_helix_angle), "mm").to_base_units().m

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.3.2. Modelagem da engrenagem do engrenamento de baixa velocidade
    </span>
</div>

A interação mecânica entre um par de engrenagens pode ser representada matematicamente por uma **rigidez de engrenamento**, conforme ilustrado no esquema abaixo. 

<p align="center"><img src="figs/rigidez_engrenamento.jpeg" width="30%"></p>

Essas propriedades podem ser tratadas como constantes ou como variáveis dependentes do tempo (ou da posição angular). Como nosso objetivo é simular com precisão a variação da rigidez conforme o contato entre os dentes evolui durante a rotação, utilizaremos a classe `GearElementTVMS` (*Time-Varying Mesh Stiffness*). 

**Nota**: Caso o objetivo fosse uma análise simplificada, onde uma rigidez média fosse suficiente para representar o comportamento do sistema, a classe base `GearElement` seria a escolha adequada.

<div style="
    width: 99%;
    display: flex;
    overflow: hidden;
    text-align: justify;
    border-radius: 10px;
    align-items: stretch;
    box-sizing: border-box;
    margin: -5px auto -5px 0px;
    border: 1px solid #cccccc;
">
    <div style="
        display: flex;
        max-width: 80px;
        color: #ffffff;
        font-weight: bold;
        padding: 14px 18px;
        align-items: center;
        white-space: nowrap;
        text-align: center;
        border-top-left-radius: 10px;
        border-bottom-left-radius: 10px;
        background: linear-gradient(90deg, #005c3b, #008542);
    ">
        Atividade<br>Prática
    </div>
    <div style="flex: 1; padding: 14px 18px; text-align: justify;">
        <p>
            Complete o código que se encontra a seguir (substituindo os <code>???</code>) 
            com base nos dados presentes na página 4 do documento <i>Special Purpose Gear Units - API 613 Fifth Edition - Data Sheet - SI Units</i>. 
        </p>
    </div>
</div>

In [ ]:
low_speed_gearbox_gear_id = Q_(772, "mm").to_base_units().m
low_speed_gearbox_gear_node = 14
low_speed_gearbox_gear_num_teeth = ???

low_speed_gearbox_gear_element = [
    rs.GearElementTVMS(
        n=low_speed_gearbox_gear_node,
        material=gearbox_gear_material,
        width=gearbox_gear_width,
        bore_diameter=low_speed_gearbox_gear_id,
        module=gearbox_gear_module,
        n_teeth=low_speed_gearbox_gear_num_teeth,
        pr_angle=gearbox_gear_pressure_angle,
        helix_angle=gearbox_gear_helix_angle,
    )
]

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px; 
        max-width: 100%;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Orientação
        </summary>
        <div style="
            padding: 20px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            display: flex;
            justify-content: center;
            overflow-x: auto;
        ">
            <img src="figs/dica_ex_2.png" style="max-width: 80%; height: auto;" alt="Datasheet">
        </div>
    </details>
</div>

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px; 
        max-width: 100%;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Ver Gabarito
        </summary>
        <div style="
            padding: 10px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            color: #333333;
            overflow-x: auto;
        ">
            <button 
                onclick="navigator.clipboard.writeText('low_speed_gearbox_gear_num_teeth = 169').then(() => { let btn = this; let oldText = btn.innerHTML; btn.innerHTML = 'Copiado!'; setTimeout(() => { btn.innerHTML = oldText; }, 2000); })"
                style="
                    margin-bottom: 15px; 
                    padding: 8px 16px; 
                    background-color: #008542; 
                    color: white; 
                    border: none; 
                    border-radius: 4px; 
                    cursor: pointer; 
                    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
                    font-weight: bold;
                    display: block;
                ">
                Copiar
            </button>

```python
low_speed_gearbox_gear_num_teeth = 169

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.3.3. Modelagem da engrenagem do engrenamento de alta velocidade
    </span>
</div>

<div style="
    width: 99%;
    display: flex;
    overflow: hidden;
    text-align: justify;
    border-radius: 10px;
    align-items: stretch;
    box-sizing: border-box;
    margin: -5px auto -5px 0px;
    border: 1px solid #cccccc;
">
    <div style="
        display: flex;
        max-width: 80px;
        color: #ffffff;
        font-weight: bold;
        padding: 14px 18px;
        align-items: center;
        white-space: nowrap;
        text-align: center;
        border-top-left-radius: 10px;
        border-bottom-left-radius: 10px;
        background: linear-gradient(90deg, #005c3b, #008542);
    ">
        Atividade<br>Prática
    </div>
    <div style="flex: 1; padding: 14px 18px; text-align: justify;">
        <p>
            Complete o código que se encontra a seguir (substituindo os <code>???</code>) 
            com base no último <code>GearElementTVMS</code> que definimos e com base nos dados presentes na página 4 do documento <i>Special Purpose Gear Units - API 613 Fifth Edition - Data Sheet - SI Units</i>. 
        </p>
    </div>
</div>

In [ ]:
high_speed_gearbox_shaft_id = Q_(110, "mm").to_base_units().m
high_speed_gearbox_gear_node = 17
high_speed_gearbox_gear_num_teeth = ???

high_speed_gearbox_gear_element = [
    rs.GearElementTVMS(
        ???
    )
]

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px; 
        max-width: 100%;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Orientação
        </summary>
        <div style="
            padding: 20px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            display: flex;
            justify-content: center;
            overflow-x: auto;
        ">
            <img src="figs/dica_ex_3.png" style="max-width: 80%; height: auto;" alt="Datasheet">
        </div>
    </details>
</div>

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px; 
        max-width: 100%;
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Ver Gabarito
        </summary>
        <div style="
            padding: 10px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            color: #333333;
            overflow-x: auto;
        ">
            <button 
                onclick="navigator.clipboard.writeText(`high_speed_gearbox_shaft_id = Q_(110, 'mm').to_base_units().m\nhigh_speed_gearbox_gear_node = 17\nhigh_speed_gearbox_gear_num_teeth = 27\n\nhigh_speed_gearbox_gear_element = [\n    rs.GearElementTVMS(\n        n=high_speed_gearbox_gear_node,\n        material=gearbox_gear_material,\n        width=gearbox_gear_width,\n        bore_diameter=high_speed_gearbox_shaft_id,\n        module=gearbox_gear_module,\n        n_teeth=high_speed_gearbox_gear_num_teeth,\n        pr_angle=gearbox_gear_pressure_angle,\n        helix_angle=gearbox_gear_helix_angle,\n    )\n]`).then(() => { let btn = this; let oldText = btn.innerHTML; btn.innerHTML = 'Copiado!'; setTimeout(() => { btn.innerHTML = oldText; }, 2000); })"
                style="
                    margin-bottom: 15px; 
                    padding: 8px 16px; 
                    background-color: #008542; 
                    color: white; 
                    border: none; 
                    border-radius: 4px; 
                    cursor: pointer; 
                    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
                    font-weight: bold;
                    display: block;
                ">
                Copiar
            </button>

```python
high_speed_gearbox_shaft_id = Q_(110, "mm").to_base_units().m
high_speed_gearbox_gear_node = 17
high_speed_gearbox_gear_num_teeth = 27

high_speed_gearbox_gear_element = [
    rs.GearElementTVMS(
        n=high_speed_gearbox_gear_node,
        material=gearbox_gear_material,
        width=gearbox_gear_width,
        bore_diameter=high_speed_gearbox_shaft_id,
        module=gearbox_gear_module,
        n_teeth=high_speed_gearbox_gear_num_teeth,
        pr_angle=gearbox_gear_pressure_angle,
        helix_angle=gearbox_gear_helix_angle,
    )
]

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.3.4. Criação dos engrenamentos (com engrenagem)
    </span>
</div>

Integrando todos os componentes definidos nas etapas anteriores (eixos, mancais e engrenagens) utilizaremos a classe `Rotor` para criar os engrenamentos, que servirão como blocos fundamentais na montagem final do sistema multirrotor.

In [24]:
low_speed_gearbox = rs.Rotor(
    low_speed_gearbox_shaft_elements,
    low_speed_gearbox_gear_element, # Inclusão do elemento de engrenagem
    low_speed_gearbox_bearing_elements,
)
low_speed_gearbox.plot_rotor(nodes=999).show()

In [25]:
high_speed_gearbox = rs.Rotor(
    high_speed_gearbox_shaft_elements,
    high_speed_gearbox_gear_element, # Inclusão do elemento de engrenagem
    high_speed_gearbox_bearing_elements,
)
high_speed_gearbox.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.4. Montagem do rotor de baixa velocidade
    </span>
</div>

Concluímos a definição de todos os elementos individuais que compõem o sistema multi-rotor: motor, compressor, acoplamentos e engrenamentos. O próximo passo consiste em combinar esses componentes em subconjuntos funcionais.

Começaremos pela montagem do rotor que integra o motor, o acoplamento e o seu respectivo engrenamento. Para realizar essa união de forma automatizada, utilizaremos a função customizada `concatenate_rotor`, definida abaixo. Esta função é responsável por reindexar os nós de cada componente, garantindo que a conectividade entre os elementos do eixo se mantenha correta após a junção.

In [26]:
def concatenate_rotor(rotor_list):
    shaft_elements = []
    disk_elements = []
    bearing_elements = []
    point_mass_elements = []

    node_offset = 0
    rotor_id = 0  # Identificador incremental para tags

    for rotor in rotor_list:
        rotor = deepcopy(rotor)

        # Reindexar elementos de eixo
        for i, el in enumerate(rotor.shaft_elements):
            el.n_l += node_offset
            el.n_r += node_offset
            el.n = el.n_l  # importante para elementos do ROSS
            el.tag = f"shaft_r{rotor_id}_{i}"
        shaft_elements.extend(rotor.shaft_elements)

        # Reindexar discos
        for i, el in enumerate(rotor.disk_elements):
            el.n += node_offset
            el.tag = f"disk_r{rotor_id}_{i}"
        disk_elements.extend(rotor.disk_elements)

        # Reindexar mancais
        for i, el in enumerate(rotor.bearing_elements):
            el.n += node_offset
            el.tag = f"bearing_r{rotor_id}_{i}"
        bearing_elements.extend(rotor.bearing_elements)

        # Reindexar massas concentradas
        for i, el in enumerate(rotor.point_mass_elements):
            el.n += node_offset
            el.tag = f"pmass_r{rotor_id}_{i}"
        point_mass_elements.extend(rotor.point_mass_elements)

        # Atualiza deslocamento para o próximo rotor
        all_nodes = [el.n_r for el in rotor.shaft_elements] + [
            el.n for el in rotor.disk_elements + rotor.bearing_elements
        ]
        node_offset = max(all_nodes)
        rotor_id += 1

    rotor_concat = rs.Rotor(
        shaft_elements=shaft_elements,
        disk_elements=disk_elements,
        bearing_elements=bearing_elements,
        point_mass_elements=point_mass_elements,
    )

    return rotor_concat

In [44]:
rotor_low_speed = concatenate_rotor([motor, low_speed_coupling, low_speed_gearbox])
rotor_low_speed.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.5. Montagem do rotor de alta velocidade
    </span>
</div>

Seguindo a mesma lógica aplicada anteriormente, utilizaremos a função `concatenate_rotor` para consolidar o segundo subconjunto do sistema. Nesta etapa, combinaremos o compressor, o seu respectivo engrenamento e o acoplamento que os interliga.

Essa operação resultará em um objeto `Rotor` único, representando toda a linha de alta velocidade. Com os dois rotores principais devidamente montados, teremos os blocos construtivos necessários para realizar o acoplamento final na caixa de engrenagens e definir o sistema multirrotor completo.

In [27]:
rotor_high_speed = concatenate_rotor(
    [high_speed_gearbox, high_speed_coupling, compressor]
)
rotor_high_speed.plot_rotor(nodes=999).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.6. Montagem do sistema multirrotor
    </span>
</div>

Com os dois subconjuntos (motor e compressor) devidamente instanciados, temos agora os blocos construtivos necessários para consolidar o modelo final. Para utilizar a classe `MultiRotor`, precisamos definir parâmetros específicos que regem a interação física e espacial entre os eixos:

* **`coupled_nodes`**: Indica os nós de cada rotor que estarão em contato por meio do engrenamento.
* **`update_mesh_stiffness`**: Uma *flag* que determina se a rigidez de engrenamento deve variar no tempo (utilizando o `GearElementTVMS`).
* **`position`**: Define a localização do segundo rotor em relação ao primeiro (neste caso, posicionado abaixo) para fins de representação. 
* **`orientation_angle`**: Especifica o ângulo de orientação relativo entre os dois rotores.

Ao instanciar este objeto, o **ROSS** acopla as matrizes de massa, rigidez e amortecimento dos dois sistemas, o que possibilita uma análise completa de toda a máquina.

In [45]:
# Utilização do método automático para determinação dos nós das engrenagens (por conta da concatenação dos rotores).

rotor_low_speed_gear_node = next(
    e.n for e in rotor_low_speed.disk_elements if isinstance(e, rs.GearElement)
)
rotor_high_speed_gear_node = next(
    e.n for e in rotor_high_speed.disk_elements if isinstance(e, rs.GearElement)
)
multirotor_coupled_node = (rotor_low_speed_gear_node, rotor_high_speed_gear_node)

print(f"Nós de acoplamento: {multirotor_coupled_node}")

multirotor = rs.MultiRotor(
    rotor_low_speed, 
    rotor_high_speed, 
    coupled_nodes=multirotor_coupled_node, 
    update_mesh_stiffness=True,
    position="below",
    orientation_angle=0,
)

multirotor.plot_rotor(nodes=999).show()

Nós de acoplamento: (57, 17)


In [ ]:
stiff_avg = multirotor.mesh.stiffness/1e9
print(f"Média de Rigidez = {stiff_avg:.2f} * 1E9 N/m")

Média de Rigidez = 3.06 N/m
Razão de Contato = 1.76


Como vimos anteriormente, a rigidez de engrenamento pode ser dinâmica e oscilar ao longo do tempo conforme o contato entre os dentes evolui. Para visualizar esse comportamento, utilizaremos o método `plot_stiffness_profile`, que nos possibilita avaliar a evolução dessa rigidez em função da posição angular. 

In [31]:
multirotor.mesh.plot_stiffness_profile(n_mesh_period=2).show()

Além do perfil de rigidez detalhado mostrado acima, o **ROSS** possibilita que adotemos uma abordagem simplificada na qual a rigidez de engrenamento varia segundo uma **onda retangular**. Esta configuração é útil para análises de sensibilidade ou para representar comportamentos de contato mais abruptos entre os dentes.

Para implementarmos essa variação, basta habilitarmos o parâmetro `square_varying_stiffness` ao instanciar a classe `MultiRotor`. Adicionalmente, podemos utilizar o parâmetro `square_stiffness_amplitude_ratio` para ajustar a amplitude dessa oscilação em relação à rigidez média do conjunto.

<div style="
    width: 99%;
    display: flex;
    overflow: hidden;
    text-align: justify;
    border-radius: 10px;
    align-items: stretch;
    box-sizing: border-box;
    margin: -5px auto -5px 0px;
    border: 1px solid #cccccc;
">
    <div style="
        display: flex;
        max-width: 80px;
        color: #ffffff;
        font-weight: bold;
        padding: 14px 18px;
        align-items: center;
        white-space: nowrap;
        text-align: center;
        border-top-left-radius: 10px;
        border-bottom-left-radius: 10px;
        background: linear-gradient(90deg, #005c3b, #008542);
    ">
        Atividade<br>Prática
    </div>
    <div style="flex: 1; padding: 14px 18px; text-align: justify;">
        <p>
            Com base no último multirrotor que definimos, atualize o código abaixo (substituindo os <code>???</code>) e execute o método responsável pela construção do gráfico que mostra a evolução da rigidez de engrenamento.
        </p>
    </div>
</div>

In [ ]:
multirotor_square_stiff = rs.MultiRotor(
    ???
    square_varying_stiffness = True,
    square_stiffness_amplitude_ratio = 0.225,
)

In [ ]:
stiff_avg = ???
print(f"Média de Rigidez = {stiff_avg:.2f} * 1E9 N/m")

In [ ]:
??? # Adicione aqui o método responsável pela construção do gráfico que mostra a evolução da rigidez de engrenamento.

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px;
        max-width: 100%; 
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Ver Gabarito
        </summary>
        <div style="
            padding: 10px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            color: #333333;
            overflow-x: auto;
        ">
            <button 
                onclick="navigator.clipboard.writeText(`multirotor_square_stiff = rs.MultiRotor(\n    rotor_low_speed, \n    rotor_high_speed, \n    coupled_nodes=multirotor_coupled_node,\n    update_mesh_stiffness=True,\n    square_varying_stiffness = True,\n    square_stiffness_amplitude_ratio = 0.225,\n    position='below',\n    orientation_angle=0,\n)\n\nstiff_avg = multirotor_square_stiff.mesh.stiffness/1e9\nprint(f'Média de Rigidez = {stiff_avg:.2f} * 1E9 N/m')\n\nmultirotor_square_stiff.mesh.plot_stiffness_profile(n_mesh_period=2).show()`).then(() => { let btn = this; let oldText = btn.innerHTML; btn.innerHTML = 'Copiado!'; setTimeout(() => { btn.innerHTML = oldText; }, 2000); })"
                style="
                    margin-bottom: 15px; 
                    padding: 8px 16px; 
                    background-color: #008542; 
                    color: white; 
                    border: none; 
                    border-radius: 4px; 
                    cursor: pointer; 
                    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
                    font-weight: bold;
                    display: block;
                ">
                Copiar
            </button>
        
```python
multirotor_square_stiff = rs.MultiRotor(
    rotor_low_speed, 
    rotor_high_speed, 
    coupled_nodes=multirotor_coupled_node,
    update_mesh_stiffness=True,
    square_varying_stiffness = True,
    square_stiffness_amplitude_ratio = 0.225,
    position="below",
    orientation_angle=0,
)

stiff_avg = multirotor_square_stiff.mesh.stiffness/1e9
print(f"Média de Rigidez = {stiff_avg:.2f} * 1E9 N/m")

multirotor_square_stiff.mesh.plot_stiffness_profile(n_mesh_period=2).show()

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 24px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.7. Análises associadas ao multirrotor
    </span>
</div>

Com o modelo do sistema multirrotor consolidado, estamos aptos a realizar as mesmas análises dinâmicas aplicáveis a rotores simples. Nesta etapa final, exploraremos a construção do **Diagrama de Campbell** e a execução da **análise modal** para identificar as frequências naturais e as velocidades críticas do sistema acoplado.

Como o foco principal deste estudo são os **modos torcionais**, utilizaremos a função `convert_6dof_to_torsional`. Esta ferramenta simplifica o modelo original (que possui 6 graus de liberdade por nó), retendo apenas os componentes torcionais. Essa redução de ordem não apenas otimiza o processamento numérico, mas também isola o comportamento de interesse, facilitando a interpretação dos resultados.

Por fim, os resultados obtidos serão comparadosconfrontados com os dados técnicos presentes em relatórios fornecidos pela **Petrobras**. Essa etapa de comparação é fundamental para a **validação do modelo**, garantindo que as premissas adotadas no **ROSS** reflitam fielmente o comportamento dinâmico do equipamento real.

In [33]:
multirotor = convert_6dof_to_torsional(multirotor)
multirotor_square_stiff = convert_6dof_to_torsional(multirotor_square_stiff)

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.7.1. Diagrama de Campbell
    </span>
</div>

Anteriormente, em nosso curso, vimos como construir o Diagrama de Campbell de um rotor simples (sem engrenamento). Felizmente, a construção do Diagrama de Campbell de um MultiRotor segue exatamente a estrutura e emprega o método `run_campbell` da mesma forma. Tendo isso em mente, visualize abaixo o código que possibilitará a construção do Diagrama de Campbell do MultiRotor que definimos agora há pouco (`multirotor`).

In [34]:
multirotor_campbell_samples = 20
multirotor_campbell_gear_ratio = multirotor.mesh.gear_ratio
multirotor_campbell_speed_range = np.linspace(
    0, Q_(15000, 'rpm').to_base_units().m, multirotor_campbell_samples
)
multirotor_campbell = multirotor.run_campbell(multirotor_campbell_speed_range)
multirotor_campbell.plot_with_mode_shape(harmonics=[1, multirotor_campbell_gear_ratio], animation=True)

C:\Users\Murillo\OneDrive - Universidade Federal de Uberlândia\Área de Trabalho\Mestrado\ENGRENAMENTO\Implementacao\ross_dev_backlash\ross\ross\rotor_assembly.py:1350: UserWarning:

Extrapolating bearing coefficients. Be careful when post-processing the results.



<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.7.2. Análise modal
    </span>
</div>

<div style="
    width: 99%;
    display: flex;
    overflow: hidden;
    text-align: justify;
    border-radius: 10px;
    align-items: stretch;
    box-sizing: border-box;
    margin: -5px auto -5px 0px;
    border: 1px solid #cccccc;
">
    <div style="
        display: flex;
        max-width: 80px;
        color: #ffffff;
        font-weight: bold;
        padding: 14px 18px;
        align-items: center;
        white-space: nowrap;
        text-align: center;
        border-top-left-radius: 10px;
        border-bottom-left-radius: 10px;
        background: linear-gradient(90deg, #005c3b, #008542);
    ">
        Atividade<br>Prática
    </div>
    <div style="flex: 1; padding: 14px 18px; text-align: justify;">
        <p>
            Com base na aula anterior, implemente abaixo a analise modal para a velocidade de 1953 RPM, criando a variável <code>multirotor_modal</code>. <br> Imprima, também, na tela os valores das frequências naturais amortecidas.
        </p>
    </div>
</div>

In [ ]:
# IMPLEMENTE AQUI

<div style="display: flex; justify-content: flex-start; width: 100%; margin: 20px 0;">
    <details 
        ontoggle="this.style.width = this.open ? 'fit-content' : '160px';"
        style="
        border: 1px solid #d4d4d4;
        border-radius: 8px;
        background-color: #ffffff;
        width: 160px; 
        max-width: 100%; 
        box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        overflow: hidden;
        transition: width 0.3s ease;
    ">
        <summary 
            style="
                cursor: pointer; 
                color: #ffffff; 
                background-color: #008542; 
                padding: 12px 16px; 
                font-weight: bold;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                list-style: none;
                outline: none;
                text-align: center;
                white-space: nowrap;
            ">
            ▶ Ver Gabarito
        </summary>
        <div style="
            padding: 10px; 
            background-color: #ffffff; 
            border-top: 1px solid #d4d4d4;
            color: #333333;
            overflow-x: auto;
        ">
            <button 
                onclick="navigator.clipboard.writeText(`multirotor_modal_speed = rs.Q_(1953, 'RPM')\nmultirotor_modal = multirotor.run_modal(speed=multirotor_modal_speed, num_modes=12)\n\nprint(f'w_d = {multirotor_modal.wd * 30 / np.pi}')`).then(() => { let btn = this; let oldText = btn.innerHTML; btn.innerHTML = 'Copiado!'; setTimeout(() => { btn.innerHTML = oldText; }, 2000); })"
                style="
                    margin-bottom: 15px; 
                    padding: 8px 16px; 
                    background-color: #008542; 
                    color: white; 
                    border: none; 
                    border-radius: 4px; 
                    cursor: pointer; 
                    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
                    font-weight: bold;
                    display: block;
                ">
                Copiar
            </button>
        
```python
multirotor_modal_speed = rs.Q_(1953, "RPM")
multirotor_modal = multirotor.run_modal(speed=multirotor_modal_speed, num_modes=12)

print(f"w_d = {multirotor_modal.wd * 30 / np.pi}")

<div style="
    display: flex;
    align-items: center;
    background: linear-gradient(90deg, #005c3b, #008542);
    border-left: 8px solid #ffd100;
    width: 99%;
    padding: 10px 14px;
    border-radius: 8px;
    box-sizing: border-box;
    margin: 0;
    margin-bottom: -10px
">
    <img 
        src="https://medproc.migalhas.com.br/https__img5.migalhas.com.br__SL__gf_base__SL__empresas__SL__miga__SL__imagens__SL__5e260e50919130eb64ff0a749224ad38ef04_br.jpg._PROC_CP75CCH31622400.jpg" 
        alt="Logo Petrobras"
        style="height: 30px; margin-right: 12px;"
    >
    <span style="
        color: #ffffff;
        font-size: 20px;
        font-weight: bold;
        font-family: Arial, sans-serif;
    ">
        2.7.3. Análise torcional
    </span>
</div>

In [36]:
multirotor_modal.plot_mode_2d(0, frequency_units="RPM").show()

<p align="center"><img src="figs/modo_1.png" width="80%"></p>

In [37]:
multirotor_modal.plot_mode_2d(1, frequency_units="RPM").show()

<p align="center"><img src="figs/modo_2.png" width="80%"></p>

In [38]:
multirotor_modal.plot_mode_2d(2, frequency_units="RPM").show()

<p align="center"><img src="figs/modo_3.png" width="80%"></p>

In [39]:
multirotor_modal.plot_mode_2d(3, frequency_units="RPM").show()

<p align="center"><img src="figs/modo_4.png" width="80%"></p>

In [40]:
multirotor_modal.plot_mode_2d(4, frequency_units="RPM").show()

<p align="center"><img src="figs/modo_5.png" width="80%"></p>

In [41]:
multirotor_modal.plot_mode_2d(5, frequency_units="RPM").show()

<p align="center"><img src="figs/modo_6.png" width="80%"></p>